In [1]:
import csv
import random
import math

In [3]:
def loadcsv(filename):
  lines = csv.reader(open(filename, "r"))
  next(lines)
  dataset = list(lines)
  for i in range(len(dataset)):
    dataset[i] = [float(x) for x in dataset[i]]
  return dataset

In [4]:
def splitdataset(dataset, splitratio):
  trainsize = int(len(dataset) * splitratio)
  trainset = []
  copy = list(dataset)
  while len(trainset) < trainsize:
    index = random.randrange(len(copy))
    trainset.append(copy.pop(index))
  return [trainset, copy]

In [5]:
def separatebyclass(dataset):
  separated = {}
  for i in range(len(dataset)):
    vector = dataset[i]
    if vector[-1] not in separated:
      separated[vector[-1]] = []
    separated[vector[-1]].append(vector)
  return separated

In [6]:
def mean(numbers):
  return sum(numbers) / float(len(numbers))

In [7]:
def stdev(numbers):
  avg = mean(numbers)
  variance = sum(
      [pow(x - avg, 2) for x in numbers]
  ) / float(len(numbers) - 1)
  return math.sqrt(variance)

In [8]:
def summarizebyclass(dataset):
  separated = separatebyclass(dataset)
  summaries = {}
  for classvalue, instances in separated.items():
    summaries[classvalue] = summarize(instances)
  return summaries

In [9]:
def calculateprobability(x, mean, stdev):
  exponent = math.exp(
      -(math.pow(x - mean, 2) /
       (2 * math.pow(stdev, 2)))
  )
  return (
      (1 / (math.sqrt(2 * math.pi) * stdev))
      * exponent
  )

In [10]:
def calculateclassprobabilities(summaries, inputvector):
  probabilities = {}
  for classvalue, classsummaries in summaries.items():
    probabilities[classvalue] = 1
    for i in range(len(classsummaries)):
      mean_value, stdev_value = classsummaries[i]
      x = inputvector[i]
      probabilities[classvalue] *= calculateprobability(
           x,
           mean_value,
           stdev_value
      )
  return probabilities

In [11]:
def predict(summaries, inputvector):
  probabilities = calculateclassprobabilities(
      summaries,
      inputvector
  )
  bestLabel = None
  bestProb = -1
  for classvalue, probability in probabilities.items():
    if bestLabel is None or probability > bestProb:
      bestProb = probability
      bestLabel = classvalue
  return bestLabel

In [12]:
def getpredictions(summaries, testset):
  predictions = []
  for i in range(len(testset)):
    result = predict(
         summaries,
         testset[i]
    )
    predictions.append(result)
  return predictions

In [13]:
def getaccuracy(testset, predictions):
  correct = 0
  for i in range(len(testset)):
    if testset[i][-1] == predictions[i]:
      correct += 1
  return (correct / float(len(testset))) * 100.0

In [22]:
def main():
  filename ="C:\\Users\\monik\\Naive-Bayes-Classification-Data.csv"
  splitratio = 0.67
  dataset = loadcsv(filename)

In [25]:
from google.colab import files
uploaded = files.upload()

Saving Naive-Bayes-Classification-Data.csv to Naive-Bayes-Classification-Data.csv


In [26]:
filename = "Naive-Bayes-Classification-Data.csv"
dataset = loadcsv(filename)

In [28]:
splitratio = 0.67
trainingset, testset = splitdataset(
    dataset,
    splitratio
)
print(
    'Split {0} rows into train={1} and test={2} rows'
    .format(
        len(dataset),
        len(trainingset),
        len(testset)
     )
)

Split 995 rows into train=666 and test=329 rows


In [31]:
import math

def summarize(dataset):
    # Calculates the mean and stdev for each attribute column
    summaries = [(mean(attribute), stdev(attribute)) for attribute in zip(*dataset)]
    del summaries[-1] # Remove the class/label column summaries
    return summaries

summaries = summarizebyclass(trainingset)
predictions = getpredictions(
    summaries,
    testset
)
accuracy = getaccuracy(
    testset,
    predictions
)
print('Accuracy of the classifier is: {0}%'.format(accuracy))

if __name__ == "__main__":
    # Note: main() relies on a hardcoded C:\ path that might not work in Colab,
    # but we will keep it as is if you wish to run it.
    pass

Accuracy of the classifier is: 93.3130699088146%
